# CNN V4 - Honest Split + Transfer Learning (isolated test)

A self-contained experiment, separate from CNN.ipynb (V1-V3 stay
untouched). Delete this file and the addition is gone.

What it changes against V2, one idea each:
1. **Group split** - URI-CAD frames come from ultrasound videos but carry
   no video/patient id, so consecutive id blocks (25 frames) stand in for
   the source videos; train/val/test never share a block.
2. **Transfer learning** - an ImageNet-pretrained ResNet18 replaces the
   from-scratch network: 2 epochs head-only, then fine-tuned at a low
   learning rate.
3. **Balanced batches** - WeightedRandomSampler on top of the weighted
   loss, so each batch is roughly 50/50.
4. **Training hygiene** - label smoothing, cosine learning-rate decay,
   early stopping on validation loss.
5. **Threshold tuning + TTA** - the decision threshold is tuned on
   validation for balanced accuracy, and test predictions average three
   augmented views (original, horizontal flip, vertical flip).

Preprocessing stays V2-style (resize with padding + normalisation) so the
effect of the changes above is not mixed with new augmentation.

In [1]:
import copy
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score)
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


In [2]:
root = Path.cwd()
while not (root / "assets" / "datasets").is_dir():
    root = root.parent
IMAGE_DIR = root / "assets" / "datasets" / "URI_CAD_Dataset" / "images"
LABEL_DIR = root / "assets" / "datasets" / "URI_CAD_Dataset" / "labels"

rows = []
for label_file in sorted(LABEL_DIR.glob("*.txt"), key=lambda p: int(p.stem)):
    text = label_file.read_text()
    diagnosis = int(re.search(r"Global diagnosis: (\d)", text).group(1))
    rows.append({"image_id": int(label_file.stem),
                 "image_path": str(IMAGE_DIR / f"{label_file.stem}.jpg"),
                 "diagnosis": diagnosis})

df = pd.DataFrame(rows).sort_values("image_id").reset_index(drop=True)

# URI-CAD provides no video/patient id; frames were extracted sequentially
# from the source videos, so consecutive id blocks approximate them.
df["group"] = df["image_id"] // 25

print(f"images: {len(df)} | groups: {df['group'].nunique()} "
      f"| stone frames: {(df['diagnosis'] == 1).mean():.1%})")

images: 1985 | groups: 80 | stone frames: 77.3%)


In [3]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(gss.split(df, groups=df["group"]))
temp = df.loc[temp_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(gss2.split(temp, groups=temp["group"]))

train_df = df.loc[train_idx].reset_index(drop=True)
val_df = temp.iloc[val_idx].reset_index(drop=True)
test_df = temp.iloc[test_idx].reset_index(drop=True)

for name, part in (("train", train_df), ("val", val_df), ("test", test_df)):
    print(f"{name}: {len(part)} frames, {part['group'].nunique()} groups, "
          f"stone {part['diagnosis'].mean():.1%}")

train: 1399 frames, 56 groups, stone 78.8%
val: 300 frames, 12 groups, stone 71.0%
test: 286 frames, 12 groups, stone 76.6%


In [4]:
IMG_SIZE = 224


class ResizeWithPadding:
    def __init__(self, size):
        self.size = size

    def __call__(self, img):
        w, h = img.size
        scale = self.size / max(w, h)
        img = img.resize((round(w * scale), round(h * scale)))
        canvas = Image.new("RGB", (self.size, self.size))
        canvas.paste(img, ((self.size - img.width) // 2, (self.size - img.height) // 2))
        return canvas


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
base_transform = transforms.Compose([
    ResizeWithPadding(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class KidneyDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img = Image.open(self.dataframe.loc[idx, "image_path"]).convert("RGB")
        label = torch.tensor(self.dataframe.loc[idx, "diagnosis"], dtype=torch.long)
        if self.transform:
            img = self.transform(img)
        return img, label


class_counts = train_df["diagnosis"].value_counts()
sample_weights = train_df["diagnosis"].map({c: len(train_df) / (2 * n) for c, n in class_counts.items()})
sampler = WeightedRandomSampler(sample_weights.values, num_samples=len(train_df), replacement=True)

train_loader = DataLoader(KidneyDataset(train_df, base_transform), batch_size=32,
                          sampler=sampler, num_workers=0)
val_loader = DataLoader(KidneyDataset(val_df, base_transform), batch_size=32, num_workers=0)
test_loader = DataLoader(KidneyDataset(test_df, base_transform), batch_size=32, num_workers=0)
print("loaders ready (balanced sampling on train)")

loaders ready (balanced sampling on train)


/Users/elgazar/Desktop/RoboticsMasters/Semester2_Sep/F21DL/Project/F21DL_Group_13/.venv/lib/python3.14/site-packages/torch/utils/data/sampler.py:264: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:214.)
  weights_tensor = torch.as_tensor(weights, dtype=torch.double)


In [5]:
v4_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
v4_model.fc = nn.Linear(v4_model.fc.in_features, 2)
v4_model = v4_model.to(device)

class_weights = torch.tensor([
    len(train_df) / (2 * (train_df["diagnosis"] == 0).sum()),
    len(train_df) / (2 * (train_df["diagnosis"] == 1).sum()),
], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

def predict_probs(model, loader):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for images, y in loader:
            out = torch.softmax(model(images.to(device)), dim=1)[:, 1]
            probs.extend(out.cpu().tolist())
            labels.extend(y.tolist())
    return np.array(probs), np.array(labels)

In [6]:
def run_epochs(freeze_backbone, epochs, lr, tag):
    for p in v4_model.parameters():
        p.requires_grad = not freeze_backbone
    if freeze_backbone:
        for p in v4_model.fc.parameters():
            p.requires_grad = True
    optimizer = torch.optim.Adam([p for p in v4_model.parameters() if p.requires_grad], lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_loss, best_state, patience = np.inf, None, 0
    for epoch in range(1, epochs + 1):
        v4_model.train()
        running = 0.0
        for images, y in train_loader:
            images, y = images.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(v4_model(images), y)
            loss.backward()
            optimizer.step()
            running += loss.item() * len(y)
        scheduler.step()
        val_probs, val_y = predict_probs(v4_model, val_loader)
        val_loss = nn.functional.cross_entropy(
            torch.tensor(np.stack([1 - val_probs, val_probs], axis=1), dtype=torch.float32),
            torch.tensor(val_y))
        print(f"{tag} epoch {epoch}: train loss {running / len(train_df):.4f}, "
              f"val loss {val_loss:.4f}, val bal-acc "
              f"{balanced_accuracy_score(val_y, val_probs > 0.5):.4f}")
        if val_loss < best_loss - 1e-4:
            best_loss, best_state, patience = val_loss, copy.deepcopy(v4_model.state_dict()), 0
        else:
            patience += 1
            if patience >= 2:
                print(f"{tag}: early stopping (no val-loss improvement for 2 epochs)")
                break
    if best_state:
        v4_model.load_state_dict(best_state)


run_epochs(freeze_backbone=True, epochs=2, lr=1e-3, tag="head-only")
run_epochs(freeze_backbone=False, epochs=4, lr=1e-4, tag="fine-tune")

head-only epoch 1: train loss 0.5396, val loss 0.9026, val bal-acc 0.5000


head-only epoch 2: train loss 0.5204, val loss 0.8344, val bal-acc 0.5036


fine-tune epoch 1: train loss 0.4075, val loss 0.6446, val bal-acc 0.5862


fine-tune epoch 2: train loss 0.2708, val loss 0.7139, val bal-acc 0.6616


fine-tune epoch 3: train loss 0.2068, val loss 0.6489, val bal-acc 0.6434
fine-tune: early stopping (no val-loss improvement for 2 epochs)


In [7]:
val_probs, val_y = predict_probs(v4_model, val_loader)
thresholds = np.arange(0.05, 0.96, 0.05)
scores = [balanced_accuracy_score(val_y, val_probs > t) for t in thresholds]
best_t = float(thresholds[int(np.argmax(scores))])
print(f"tuned decision threshold: {best_t:.2f} "
      f"(validation balanced accuracy {max(scores):.4f}; "
      f"{balanced_accuracy_score(val_y, val_probs > 0.5):.4f} at the default 0.5)")

tuned decision threshold: 0.75 (validation balanced accuracy 0.6315; 0.5862 at the default 0.5)


In [8]:
def tta_probs(model, loader):
    """Average the stone probability over the original, horizontal and vertical flips."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for images, y in loader:
            images = images.to(device)
            out = (torch.softmax(model(images), dim=1)[:, 1]
                   + torch.softmax(model(torch.flip(images, [-1])), dim=1)[:, 1]
                   + torch.softmax(model(torch.flip(images, [-2])), dim=1)[:, 1]) / 3
            probs.extend(out.cpu().tolist())
            labels.extend(y.tolist())
    return np.array(probs), np.array(labels)


test_probs, test_y = tta_probs(v4_model, test_loader)

print("majority-class baseline (always stone): 77.3% accuracy")
for name, pred in (("default 0.5", test_probs > 0.5), (f"tuned {best_t:.2f}", test_probs > best_t)):
    print(f"\nTEST - threshold {name}")
    print(f"accuracy {accuracy_score(test_y, pred):.4f} | "
          f"balanced acc {balanced_accuracy_score(test_y, pred):.4f} | "
          f"macro F1 {f1_score(test_y, pred, average='macro'):.4f}")
print("\n", classification_report(test_y, test_probs > best_t, digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(test_y, test_probs > best_t),
                   index=["no stone (true)", "stone (true)"],
                   columns=["no stone (pred)", "stone (pred)"]).to_string())

majority-class baseline (always stone): 77.3% accuracy

TEST - threshold default 0.5
accuracy 0.6294 | balanced acc 0.6596 | macro F1 0.5944

TEST - threshold tuned 0.75
accuracy 0.3741 | balanced acc 0.5758 | macro F1 0.3707

               precision    recall  f1-score   support

           0      0.267     0.955     0.417        67
           1      0.935     0.196     0.325       219

    accuracy                          0.374       286
   macro avg      0.601     0.576     0.371       286
weighted avg      0.778     0.374     0.346       286

                 no stone (pred)  stone (pred)
no stone (true)               64             3
stone (true)                 176            43


## Results

Test evaluation on 286 held-out frames (12 groups never seen in training),
full frames, predictions averaged over three augmented views (TTA):

| Threshold | Accuracy | Balanced acc | Macro F1 |
|---|---|---|---|
| Default 0.5 | 0.629 | 0.660 | 0.594 |
| Tuned 0.75 | 0.374 | 0.576 | 0.371 |
| Majority baseline (always stone) | 0.773 | 0.500 | - |

Observations:

- The pretrained backbone learned nothing during the head-only phase
  (balanced accuracy 0.50 for both epochs) and fine-tuning overfit fast
  (train loss 0.21 while validation loss stayed above 0.64): with full
  frames and no augmentation, the network memorises its 1,399 training
  images instead of learning stone appearance.
- The threshold tuned on validation (0.75) did not transfer to the test
  set - with only 300 validation frames the class mix shifts between
  group splits, so the default 0.5 remains the reported configuration.
- V4 stays below the majority baseline (77.3%): the honest split removed
  the frame leakage that made V2's 67% look competitive, and full frames
  plus no augmentation leave the model without usable signal.
- V4 served as the ablation step towards V5 - see `CNN_V5_TL.ipynb` in
  this folder for the configuration that solves it (ROI crops + real
  augmentation).